## filter the Label Studio annotations

Label Studio only allows to output all the annotations (so also the different annotations for the same text). Firt you need to make sure that you only have those annotations that you need to train the model. In this code snippet, we filter the annotations to only include those that are relevant to us (the most recent and validated annotations). We do this by filtering on the text "id" and "annotator".

In [2]:
import json

# Load your JSON data from the specified input file
with open("LS_labeled_BHF.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Define ID ranges
id_ranges = [
    range(11317, 11383),  # 11317–11382
    range(11417, 11517),  # 11417–11516
    range(11517, 11553)   # 11517–11552
]

# Define allowed annotators
allowed_annotators = {17, 18, 19}

# Filter the data
filtered_data = [
    item for item in data
    if any(item["id"] in r for r in id_ranges)
    and item.get("annotator") in allowed_annotators
]

# Save the filtered JSON to the specified output file
with open("LS_labeled_BHF_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Filtered {len(filtered_data)} items out of {len(data)} total.")

Filtered 190 items out of 927 total.


In [2]:
# reducing the dataset

import json
import random

# Paths
input_path = "LS_labeled_BHF_filtered.json"


# Load the JSON file
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Set the fraction of entries to keep (0.5 = 50%)
fraction_to_keep = 0.5
num_to_keep = int(len(data) * fraction_to_keep)

# Randomly select entries to keep
reduced_data = random.sample(data, num_to_keep)

output_path = f"LS_labeled_BHF_reduced_{fraction_to_keep}.json"

# Save the reduced JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(reduced_data, f, ensure_ascii=False, indent=2)

print(f"Reduced from {len(data)} entries to {len(reduced_data)} entries.")

Reduced from 190 entries to 95 entries.


## Label Studio to GLiNER2 Training Data Conversion
This code is used to convert the human labeled entities in Label Studio to a format that can be used for training GLiNER2. It reads the labeled data from Label Studio, extracts the relevant information, and creates a new JSON file that can be used for training.
It follows the recommended format:
```json
{"input": "text to process", "output": {"schema_definition": "with_annotations"}}
```

In [3]:
import json
from collections import defaultdict


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def convert_to_gliner2_format(annotation_data, schema):
    """
    Converts Label Studio-style annotations to GLiNER2 format.
    Only uses labels defined inside schema["entities"].
    """

    # ✅ Only take actual NER entity types
    entity_types = list(schema["entities"].keys())

    gliner_data = []

    for item in annotation_data:
        text = item["text"]
        annotations = item.get("label", [])

        entities_dict = defaultdict(list)

        # Collect entities
        for ann in annotations:
            entity_text = ann["text"].strip()
            labels = ann.get("labels", [])

            for label in labels:
                # ✅ Only include labels that exist in schema
                if label in entity_types:
                    entities_dict[label].append(entity_text)

        # Deduplicate while preserving order
        for label in entities_dict:
            seen = set()
            deduped = []
            for ent in entities_dict[label]:
                if ent not in seen:
                    deduped.append(ent)
                    seen.add(ent)
            entities_dict[label] = deduped

        # Ensure ALL schema entity types are present (even if empty)
        for entity_type in entity_types:
            if entity_type not in entities_dict:
                entities_dict[entity_type] = []

        formatted_item = {
            "text": text,
            "entities": dict(entities_dict)
            }


        gliner_data.append(formatted_item)

    return gliner_data


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    annotation_file = "LS_labeled_BHF_reduced_0.5.json"
    schema_file = "gliner_schema_BelHisFirm.json"
    output_file = "gliner2_training_data_BHF.json"

    annotations = load_json(annotation_file)
    schema = load_json(schema_file)

    gliner_data = convert_to_gliner2_format(annotations, schema)
    save_json(gliner_data, output_file)

    print(f"Converted {len(gliner_data)} examples.")

Converted 95 examples.


## finetuning GLiNER2 with LoRA
Now that we have the training data in the correct format, we can use it to fine-tune GLiNER2 specifically for our use case.

In [4]:
!pip install gliner2

## Step 1: prepare Domain-Specific Data
We have already prepared our training data in the previous step. The file `gliner2_training_data.json` contains the text and corresponding entities in the format required for GLiNER2 training. Now we can load this data and use it for fine-tuning the model.

In [5]:
from gliner2.training.data import InputExample

with open("gliner2_training_data_BHF.json", "r", encoding="utf-8") as f:
    data = json.load(f)

train_data = [
    InputExample(text=item["text"], entities=item["entities"])
    for item in data
]

C:\Users\Heike\PycharmProjects\ghentcdh-glinerv2-tutorial\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Set Up LoRA Training
Next, we will set up the LoRA training configuration. This involves specifying the adaptor we want to train, the training parameters and the LoRA settings.

In [6]:
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

# LoRA configuration
config = TrainingConfig(
    output_dir="./BelHisFirm_adaptor",       # change this to your desired output directory
    experiment_name="BelHisFirm",            # change this to your desired experiment name

    # Training parameters
    num_epochs=10,
    batch_size=8,
    gradient_accumulation_steps=2,
    encoder_lr=1e-5,
    task_lr=5e-4,

    # LoRA settings
    use_lora=True,                              # Enable LoRA
    lora_r=8,                                   # Rank (4, 8, 16, 32)
    lora_alpha=16.0,                           # Scaling factor (usually 2*r)
    lora_dropout=0.0,                          # Dropout for LoRA layers
    lora_target_modules=["encoder"],           # Apply to all encoder layers (query, key, value, dense)
    save_adapter_only=True,                    # Save only adapter (not full model)

    # Optimization
    eval_strategy="epoch",  # Evaluates and saves at end of each epoch
    eval_steps=500,  # Used when eval_strategy="steps"
    logging_steps=50,
    fp16=True,  # Use mixed precision if GPU available
)

## Step 3: Train the LoRA Adaptor
Now we can initialize the GLiNER2 model and the trainer, and start the training process

In [ ]:
# Load base model
base_model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

# Create trainer
trainer = GLiNER2Trainer(model=base_model, config=config)

# Train adapter
trainer.train(train_data=train_data)

# Adapter automatically saved to ./Hagiography_adapter/final/

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


Mixed precision disabled on CPU
2026-03-13 14:04:11 - INFO - gliner2.training.trainer - Setting up LoRA for parameter-efficient fine-tuning...
2026-03-13 14:04:11 - INFO - gliner2.training.trainer - Froze all model parameters for LoRA training
2026-03-13 14:04:11 - INFO - gliner2.training.lora - Applied LoRA to 72 layers
2026-03-13 14:04:11 - INFO - gliner2.training.trainer - LoRA setup complete: 1,327,104 trainable params out of 209,803,925 total (0.63%)


🔧 LoRA Configuration
Enabled            : True
Rank (r)           : 8
Alpha              : 16.0
Scaling (α/r)      : 2.0000
Dropout            : 0.0
Target modules     : encoder
LoRA layers        : 72
----------------------------------------------------------------------
Trainable params   : 1,327,104 / 209,803,925 (0.63%)
Memory savings     : ~99.4% fewer gradients


Validating records: 100%|██████████| 95/95 [00:00<00:00, 2014.03record/s]
2026-03-13 14:04:11 - INFO - gliner2.training.trainer - Optimizer: LoRA params only = 144, LR=0.0005
C:\Users\Heike\PycharmProjects\ghentcdh-glinerv2-tutorial\venv\lib\site-packages\gliner2\training\trainer.py:900: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)
2026-03-13 14:04:11 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-03-13 14:04:11 - INFO - gliner2.training.trainer -   Num examples = 95
2026-03-13 14:04:11 - INFO - gliner2.training.trainer -   Num epochs = 10
2026-03-13 14:04:11 - INFO - gliner2.training.trainer -   Batch size = 8
2026-03-13 14:04:11 - INFO - gliner2.training.trainer -   Gradient accumulation steps = 2
2026-03-13 14:04:11 - INFO - gliner2.training.trainer -   Effective batch size = 16
2026-03-13 14:04:11 - INFO - gliner2.training.tra

## Step 4: load and test the trained adapter
After training, we can load the trained adapter and test it on some example sentences to see how

In [6]:
## Load the trained adapter

model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
model.load_adapter("./Hagiography_adaptor/final")

# Check if adapter is loaded
if model.has_adapter:
    print("Adapter is loaded")

# Get adapter configuration
config = model.adapter_config
print(f"LoRA rank: {config.lora_r}")

2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-06 11:05:50 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/r

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


2026-03-06 11:05:55 - INFO - gliner2.training.lora - Loaded 144 LoRA tensors from Hagiography_adaptor\final\adapter_weights.safetensors
2026-03-06 11:05:55 - INFO - gliner2.training.lora - Applied LoRA to 72 layers
2026-03-06 11:05:55 - INFO - gliner2.training.lora - Loaded LoRA adapter from Hagiography_adaptor\final


Adapter is loaded
LoRA rank: 8


In [9]:
from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)
SCHEMA_CONFIG_PATH = "./gliner_schema_hagiographics.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

extractor = model

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

text = """
[1] Eo namque tempore, quo fabrica altaris ecclesiæ nostræ agebatur, contigit adesse ibi quemdam Joseph nomine, qui ita curvus erat, ut supra genua incumbens caput sursum erigere non posset: sed nec penitus immobile id uno in loco tenere diutius valebat. Qualiter autem illi acciderit, referebat: quia dum esset solus positus cum suo equo in campo vicini sui, ejusque herbam injuste depasceret, a fulgure ignis divinitus percussus ita exustus est, ut mox capite deposito curvus redderetur.
"""

results = results = extractor.extract(text, schema, threshold=0.1, include_confidence=True, include_spans=True, format_results=False)

print(json.dumps(results, indent=2, ensure_ascii=False))


{
  "entities": [
    {
      "person": [
        {
          "text": "Joseph",
          "confidence": 0.9684553146362305,
          "start": 98,
          "end": 104
        }
      ],
      "group": [],
      "institution": [
        {
          "text": "altaris ecclesiæ",
          "confidence": 0.6541092991828918,
          "start": 36,
          "end": 52
        }
      ],
      "place": [
        {
          "text": "campo",
          "confidence": 0.8170961141586304,
          "start": 343,
          "end": 348
        }
      ],
      "object": [],
      "divine_entity": [],
      "text_title": []
    }
  ]
}


## Batch processing with the trained adapter
Now that we have the trained adapter, we can use it to process a batch of texts. We will load a set of example texts, run them through the model, and print the extracted entities

### Step 1: Load Batch of Texts

In [11]:
import glob
import os

# Define the folder containing your .txt files
input_folder = 'Hagiographics_example_texts'

# Find all .txt files in the folder
txt_files = glob.glob(os.path.join(input_folder, '*.txt'))

# Load all texts and keep track of filenames
texts_batch = []
filenames = []

for txt_file in txt_files:
    with open(txt_file, 'r', encoding='utf-8') as f:
        text = f.read()
        texts_batch.append(text)
        filenames.append(os.path.basename(txt_file))

print(f"✓ Loaded {len(texts_batch)} text files from {input_folder}")
for i, filename in enumerate(filenames):
    print(f"  {i+1}. {filename} ({len(texts_batch[i])} characters)")

✓ Loaded 218 text files from Hagiographics_example_texts
  1. BHL2605 - clean_1.txt (1015 characters)
  2. BHL2605 - clean_2.txt (660 characters)
  3. BHL2605 - clean_3.txt (970 characters)
  4. BHL2605 - clean_4.txt (1548 characters)
  5. BHL2605 - clean_5.txt (1465 characters)
  6. BHL2605 - clean_6.txt (1050 characters)
  7. BHL2605 - clean_7.txt (1082 characters)
  8. BHL3563-3564 - clean_1.txt (13 characters)
  9. BHL3563-3564 - clean_10.txt (1281 characters)
  10. BHL3563-3564 - clean_11.txt (1284 characters)
  11. BHL3563-3564 - clean_12.txt (710 characters)
  12. BHL3563-3564 - clean_13.txt (1305 characters)
  13. BHL3563-3564 - clean_14.txt (1157 characters)
  14. BHL3563-3564 - clean_15.txt (830 characters)
  15. BHL3563-3564 - clean_16.txt (413 characters)
  16. BHL3563-3564 - clean_17.txt (867 characters)
  17. BHL3563-3564 - clean_18.txt (787 characters)
  18. BHL3563-3564 - clean_19.txt (822 characters)
  19. BHL3563-3564 - clean_2.txt (666 characters)
  20. BHL3563-3564 

### Step 2: Process Batch with Trained Adapter

In [12]:
# !! first make sure to load the trained adapter before running this code, as shown in the previous step !!
extractor = model

# Load schema from config file (includes both entities and relations)
schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

# Extract entities and relations from all texts
results_batch = []

print(f"Processing {len(texts_batch)} documents...")
for i, text in enumerate(texts_batch):
    print(f"  Processing {filenames[i]}...", end=" ")

    # Extract BOTH entities and relations with confidence scores and spans
    result = extractor.extract(
        text,
        schema,
        include_confidence=True,
        include_spans=True,
        format_results=False
    )

    results_batch.append(result)

    # Count total entities and relations found
    entities = result.get('entities', {})

    # IMPORTANT: GLiNER2 outputs relations as TOP-LEVEL KEYS, not nested under "relations"
    # Extract all relation types from the schema config
    schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)
    relation_types = set(schema_config.get('relations', {}).keys())

    # Gather all relations from top-level keys that match relation types
    all_relations = {}
    for rel_type in relation_types:
        if rel_type in result and result[rel_type]:
            all_relations[rel_type] = result[rel_type]

    # Count entities
    if isinstance(entities, dict):
        total_entities = sum(len(ents) for ents in entities.values())
    elif isinstance(entities, list):
        flat_count = 0
        for item in entities:
            if isinstance(item, dict) and any(isinstance(v, list) for v in item.values()):
                flat_count += sum(len(v) for v in item.values() if isinstance(v, list))
            elif isinstance(item, dict):
                flat_count += 1
            else:
                flat_count += 1
        total_entities = flat_count
    else:
        total_entities = 0

    # Count relations from all gathered relation types
    total_relations = sum(len(rels) for rels in all_relations.values() if isinstance(rels, list))

    print(f"✓ Found {total_entities} entities, {total_relations} relations")

print(f"\n✓ Extraction complete! Processed {len(results_batch)} documents")
print(schema)

Processing 218 documents...
  Processing BHL2605 - clean_1.txt... ✓ Found 11 entities, 0 relations
  Processing BHL2605 - clean_2.txt... ✓ Found 8 entities, 0 relations
  Processing BHL2605 - clean_3.txt... ✓ Found 14 entities, 0 relations
  Processing BHL2605 - clean_4.txt... ✓ Found 25 entities, 0 relations
  Processing BHL2605 - clean_5.txt... ✓ Found 22 entities, 0 relations
  Processing BHL2605 - clean_6.txt... ✓ Found 16 entities, 0 relations
  Processing BHL2605 - clean_7.txt... ✓ Found 15 entities, 0 relations
  Processing BHL3563-3564 - clean_1.txt... ✓ Found 0 entities, 0 relations
  Processing BHL3563-3564 - clean_10.txt... ✓ Found 11 entities, 0 relations
  Processing BHL3563-3564 - clean_11.txt... ✓ Found 14 entities, 0 relations
  Processing BHL3563-3564 - clean_12.txt... ✓ Found 4 entities, 0 relations
  Processing BHL3563-3564 - clean_13.txt... ✓ Found 6 entities, 0 relations
  Processing BHL3563-3564 - clean_14.txt... ✓ Found 5 entities, 0 relations
  Processing BHL356

### Step 3: convert results to Label Studio format for evaluation

In [13]:
# Reload the conversion module to pick up changes
import importlib
import gliner_to_labelstudio
importlib.reload(gliner_to_labelstudio)
from gliner_to_labelstudio import batch_convert_gliner_to_labelstudio

print("✓ Conversion module reloaded")

✓ Conversion module reloaded


In [14]:
# Convert all GLiNER2 entities to Label Studio format (relations are ignored)
ls_tasks_batch = batch_convert_gliner_to_labelstudio(
    gliner_outputs=results_batch,
    texts=texts_batch,
    model_version="gliner2-multi-v1"
)

print(f"✓ Converted {len(ls_tasks_batch)} documents to Label Studio format")
print("\nConversion Summary:")
for i, (filename, task) in enumerate(zip(filenames, ls_tasks_batch)):
    num_predictions = len(task['predictions'][0]['result'])
    avg_confidence = task['predictions'][0]['score']

    # Count entities vs relations in the result
    result_items = task['predictions'][0]['result']
    num_entities = sum(1 for item in result_items if item.get('type') == 'labels')
    num_relations = sum(1 for item in result_items if item.get('type') == 'relation')

    print(f"  {i+1}. {filename}:")
    print(f"      {num_entities} entities, {num_relations} relations")
    print(f"      avg confidence: {avg_confidence:.2%}")

✓ Converted 218 documents to Label Studio format

Conversion Summary:
  1. BHL2605 - clean_1.txt:
      11 entities, 0 relations
      avg confidence: 72.13%
  2. BHL2605 - clean_2.txt:
      8 entities, 0 relations
      avg confidence: 65.46%
  3. BHL2605 - clean_3.txt:
      14 entities, 0 relations
      avg confidence: 73.91%
  4. BHL2605 - clean_4.txt:
      25 entities, 0 relations
      avg confidence: 73.99%
  5. BHL2605 - clean_5.txt:
      22 entities, 0 relations
      avg confidence: 82.75%
  6. BHL2605 - clean_6.txt:
      16 entities, 0 relations
      avg confidence: 76.15%
  7. BHL2605 - clean_7.txt:
      15 entities, 0 relations
      avg confidence: 81.43%
  8. BHL3563-3564 - clean_1.txt:
      0 entities, 0 relations
      avg confidence: 0.00%
  9. BHL3563-3564 - clean_10.txt:
      11 entities, 0 relations
      avg confidence: 66.08%
  10. BHL3563-3564 - clean_11.txt:
      14 entities, 0 relations
      avg confidence: 66.62%
  11. BHL3563-3564 - clean_12.txt:


In [15]:
#save batch output to downloads

import json
from datetime import datetime

# Create output filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = rf'./labelstudio_batch_{timestamp}.json'

# Save batch to JSON file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(ls_tasks_batch, f, indent=2, ensure_ascii=False)

print(f"✓ Batch export complete!")
print(f"✓ Saved {len(ls_tasks_batch)} tasks to:")
print(f"  {output_file}")
print(f"\nFile size: {os.path.getsize(output_file) / 1024:.2f} KB")
print(f"\nYou can now import this file into Label Studio:")
print(f"  1. Go to your Label Studio project")
print(f"  2. Settings → Import")
print(f"  3. Upload {os.path.basename(output_file)}")
print(f"  4. Select 'Import as pre-annotations'")

✓ Batch export complete!
✓ Saved 218 tasks to:
  ./labelstudio_batch_20260306_114751.json

File size: 1065.76 KB

You can now import this file into Label Studio:
  1. Go to your Label Studio project
  2. Settings → Import
  3. Upload labelstudio_batch_20260306_114751.json
  4. Select 'Import as pre-annotations'
